In [ ]:
# %%
# Configuration
BATCH_SCENARIOS = [126, 245, 370, 585]
TIME_PERIODS = [(2020, 2059), (2060, 2099)]

SHAPEFILE_PATH = "../shape/Brazos_catchments.shp"
EXPORT_FORMAT = 'parquet'
MODEL = "NorESM2-MM"
EXPERIMENT = "DBCCA_Daymet"

N_JOBS = 12  # Number of parallel workers (adjust to your CPU core count)

# %%
import os
import tempfile
import uuid
from pathlib import Path
import pandas as pd
import xarray as xr
import geopandas as gpd
from exactextract import exact_extract
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# %%
# Global variables (initialized per worker)
_gdf_cache = None
_bounds_cache = None

def get_gdf(shapefile_path):
    """Load and cache the GeoDataFrame."""
    global _gdf_cache, _bounds_cache
    if _gdf_cache is None:
        gdf = gpd.read_file(shapefile_path)
        _gdf_cache = gdf.to_crs(epsg=4326)
        _bounds_cache = _gdf_cache.total_bounds
    return _gdf_cache, _bounds_cache


def process_year_nc(nc_file, shapefile_path):
    """Process single NetCDF file."""
    tmp_path = None
    
    try:
        # Load GeoDataFrame in each worker
        gdf_4326, bounds = get_gdf(shapefile_path)
        
        ds = xr.open_dataset(nc_file)
        da = ds['prcp'].rio.write_crs('EPSG:4326')
        da_clipped = da.rio.clip_box(*bounds)
        
        if da_clipped.lat[0] < da_clipped.lat[-1]:
            da_clipped = da_clipped.sortby('lat', ascending=False)
        
        da_multiband = da_clipped.transpose('time', 'lat', 'lon').rename(time='band')
        
        # Create unique temporary file path
        tmp_path = os.path.join(tempfile.gettempdir(), f"raster_{uuid.uuid4().hex}.tif")
        
        try:
            da_multiband.rio.to_raster(tmp_path)
            df_zonal = exact_extract(tmp_path, gdf_4326, ['mean'],
                                     include_cols=['FEATUREID'], output='pandas')
        finally:
            if tmp_path and os.path.exists(tmp_path):
                try:
                    os.remove(tmp_path)
                except:
                    pass
        
        time_strs = pd.to_datetime(da_clipped.time.values).strftime('%Y-%m-%d')
        rename_dict = {f'band_{i+1}_mean': time_strs[i] for i in range(len(time_strs))}
        df_zonal = df_zonal.rename(columns=rename_dict)
        df_zonal = df_zonal.set_index('FEATUREID').T
        df_zonal.columns = df_zonal.columns.astype(str)
        df_zonal.index = pd.to_datetime(df_zonal.index)
        
        ds.close()
        return {'status': 'success', 'data': df_zonal, 'file': nc_file}
        
    except Exception as e:
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except:
                pass
        return {'status': 'error', 'error': str(e), 'file': nc_file}


def process_scenario_joblib(nc_dir, start_year, end_year, scenario, shapefile_path, n_jobs=N_JOBS):
    """Process all years with joblib."""
    print(f"Processing years {start_year}-{end_year} with {n_jobs} workers...")
    
    # Build list of NetCDF files for the year range
    nc_files = []
    for year in range(start_year, end_year + 1):
        nc_file = os.path.join(
            nc_dir,
            f"{MODEL}_ssp{scenario}_r1i1p1f1_{EXPERIMENT}_VIC4_prcp_{year}.nc"
        )
        if os.path.exists(nc_file):
            nc_files.append(nc_file)
    
    if not nc_files:
        print("No files found")
        return None
    
    print(f"Found {len(nc_files)} files")
    
    # Parallel processing with joblib
    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=5)(
        delayed(process_year_nc)(nc_file, shapefile_path) 
        for nc_file in nc_files
    )
    
    # Separate successful results from errors
    dfs = []
    errors = []
    for r in results:
        if r['status'] == 'success':
            dfs.append(r['data'])
        else:
            errors.append((r['file'], r['error']))
    
    if errors:
        print(f"\nErrors in {len(errors)} files:")
        for f, e in errors[:5]:
            print(f"  {os.path.basename(f)}: {e}")
        if len(errors) > 5:
            print(f"  ... and {len(errors)-5} more")
    
    if not dfs:
        print("No valid data")
        return None
    
    # Merge all yearly DataFrames and export
    df = pd.concat(dfs).sort_index()
    df.index.name = 'date'
    
    print(f"\nCombined: {df.shape[0]:,} days × {df.shape[1]:,} locations")
    print(f"Range: {df.index[0]} to {df.index[-1]}")
    
    basename = f"{MODEL}_ssp{scenario}_r1i1p1f1_{EXPERIMENT}_prcp_{start_year}_{end_year}_daily"
    filepath = f"{basename}.parquet"
    df.to_parquet(filepath, compression='snappy', index=True)
    
    print(f"Exported: {filepath} ({os.path.getsize(filepath)/1e6:.1f} MB)\n")
    return df


# %%
# Verify shapefile exists
print(f"Loading shapefile: {SHAPEFILE_PATH}")
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"Loaded {len(gdf)} catchments\n")

# %%
# Process all scenarios
results = {}

total_tasks = len(BATCH_SCENARIOS) * len(TIME_PERIODS)
print(f"Processing {len(BATCH_SCENARIOS)} scenarios × {len(TIME_PERIODS)} periods = {total_tasks} total")
print(f"Workers: {N_JOBS}\n")

for scenario in BATCH_SCENARIOS:
    for start_year, end_year in TIME_PERIODS:
        print(f"\n{'='*70}")
        print(f"SSP{scenario}: {start_year}-{end_year}")
        print(f"{'='*70}")

        nc_dir = f"{MODEL}_ssp{scenario}_r1i1p1f1_{EXPERIMENT}_prcp"  # Directory created by 01_download_data.ipynb
        
        df = process_scenario_joblib(
            nc_dir, start_year, end_year, scenario, 
            SHAPEFILE_PATH, n_jobs=N_JOBS
        )
        
        if df is not None:
            results[f'ssp{scenario}_{start_year}_{end_year}'] = df
            print(f"✓ Completed")
        else:
            print(f"✗ Processing failed")

print(f"\n{'='*70}")
print(f"COMPLETE: {len(results)}/{total_tasks} datasets")
print(f"{'='*70}")